<a href="https://colab.research.google.com/github/dokunoale/chagas/blob/feature%2FCNN/notebooks/CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone --branch feature/CNN --single-branch https://github.com/dokunoale/chagas.git
%cd chagas
# Carica librerie installate
!pip install wfdb -q
!pip install gdown -q

# Aggiungi solo la root del progetto (src)
import sys
sys.path.append('/content/chagas/src')

# Importa tutto dai moduli
from preprocessing import tf_dataset_loader
from models import utils

# Importa simboli specifici (se vuoi)
from preprocessing.tf_dataset_loader import *
from models.utils import *

Cloning into 'chagas'...
remote: Enumerating objects: 147, done.
remote: Counting objects: 100% (42/42), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 147 (delta 26), reused 19 (delta 17), pack-reused 105 (from 1)
Receiving objects: 100% (147/147), 58.02 KiB | 2.32 MiB/s, done.
Resolving deltas: 100% (55/55), done.
/content/chagas
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.8/163.8 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 26.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.0 which is incompatible.
dask-cudf-cu12 25.2.2 requires pandas<2.2.4dev0,>=2.0, but you have pandas 2.3.0 which is incompatible.
cudf-cu12 25.2.1 requires pandas<2

In [3]:
import gdown

url = "https://drive.google.com/file/d/1RrxSAqML5xLWFSgilWLBv_X_H1vd04GG/view?usp=drive_link"
gdown.download(url, "dataset.zip", quiet=False, fuzzy=True)
!unzip -q dataset.zip -d ./dataset

Downloading...
From (original): https://drive.google.com/uc?id=1RrxSAqML5xLWFSgilWLBv_X_H1vd04GG
From (redirected): https://drive.google.com/uc?id=1RrxSAqML5xLWFSgilWLBv_X_H1vd04GG&confirm=t&uuid=824966b0-8884-4136-83c9-5a7aa66bd36b
To: /content/chagas/dataset.zip
100%|██████████| 332M/332M [00:03<00:00, 102MB/s]


In [4]:
#Carichiamo su array np
X_train_positive, y_train_positive = tf_dataset_loader.load_dataset('/content/chagas/dataset/splitted_dataset/train/positives')
X_train_negative, y_train_negative = tf_dataset_loader.load_dataset('/content/chagas/dataset/splitted_dataset/train/negatives')

X_val_positive, y_val_positive = tf_dataset_loader.load_dataset('/content/chagas/dataset/splitted_dataset/val/positives')
X_val_negative, y_val_negative = tf_dataset_loader.load_dataset('/content/chagas/dataset/splitted_dataset/val/negatives')

X_test_positive, y_test_positive = tf_dataset_loader.load_dataset('/content/chagas/dataset/splitted_dataset/test/positives')
X_test_negative, y_test_negative = tf_dataset_loader.load_dataset('/content/chagas/dataset/splitted_dataset/test/negatives')

In [5]:
print(f"TRAIN: {X_train_positive.shape[0]} positivi, {X_train_negative.shape[0]} negativi")
print(f"VAL: {X_val_positive.shape[0]} positivi, {X_val_negative.shape[0]} negativi")
print(f"TEST: {X_test_positive.shape[0]} positivi, {X_test_negative.shape[0]} negativi")

TRAIN: 2090 positivi, 2110 negativi
VAL: 300 positivi, 300 negativi
TEST: 610 positivi, 590 negativi


In [6]:
X_train, y_train = tf_dataset_loader.concatenate_and_shuffle((X_train_positive, y_train_positive), (X_train_negative, y_train_negative))
X_val, y_val = tf_dataset_loader.concatenate_and_shuffle((X_val_positive, y_val_positive), (X_val_negative, y_val_negative))
X_test, y_test = tf_dataset_loader.concatenate_and_shuffle((X_test_positive, y_test_positive), (X_test_negative, y_test_negative))

In [7]:
print(f"TRAIN SHAPE: {X_train.shape} - VALIDATION SHAPE: {X_val.shape} - TEST SHAPE: {X_test.shape}")

TRAIN SHAPE: (4200, 2800, 12) - VALIDATION SHAPE: (600, 2800, 12) - TEST SHAPE: (1200, 2800, 12)


In [8]:
y_train = y_train.astype(int)
y_val = y_val.astype(int)
y_test = y_test.astype(int)

# MODELLO CNN

In [9]:
from models import layers

In [10]:
!git pull origin feature/CNN
%cd /content/chagas/src/models
from models import build_CNN
from build_CNN import build_ecg_model_with_spectrogram
import importlib
import build_CNN

importlib.reload(build_CNN)

from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.metrics import AUC
from sklearn.utils import class_weight

From https://github.com/dokunoale/chagas
 * branch            feature/CNN -> FETCH_HEAD
Already up to date.
/content/chagas/src/models


In [11]:
model = build_CNN.build_ecg_model_with_spectrogram ()

#compiliamo il modello
model.compile(optimizer='adam',
              loss=BinaryCrossentropy(),
              metrics=['accuracy', AUC(name='auc')])

#addestriamo il modello
callback = make_callback("1_CNN")

history1 = model.fit(X_train, y_train,
                    validation_data=(X_val, y_val),
                    epochs=20,
                    batch_size=32,
                    callbacks=callback)


NameError: name 'np' is not defined